# Peeking, priced

Somebody is going to look at the results before the study ends. They should — a treatment
that is hurting people should not run to full enrollment. But a study that *may* stop on what
it sees is not a study whose fixed-sample error rate applies, and the gap is not small: five
unadjusted looks at α = 0.05 spend about 14%, not 5%.

The fix is old and well understood. What this module adds is that every number in it is
*computed for the rule you actually wrote down* — including the harm boundary your monitoring
committee stated as a sentence about a posterior, and the futility boundary everyone agrees
to ignore.

A study that may stop on what it sees is not a study whose fixed-sample error rate
applies. `axiom.design.sequential` supplies three things and nothing else:

1. **boundaries** — thresholds on the interim statistic, one per look, from a classic
   shape (`pocock`, `obrien_fleming`), an alpha-spending function (`alpha_spending`), or
   a posterior-probability rule stated in words (`harm_boundary`);
2. **crossing probabilities** — `crossing_probabilities` / `operating_characteristics`
   integrate the canonical joint distribution exactly, so the error a rule *actually*
   spends is a computed number rather than a claimed one;
3. **the decision** — `monitor` walks realized statistics against a `StoppingRule` and
   returns the look it stopped at, plus a `LedgerLine` that names the bias of the
   estimate reported there.

The monitoring statistic is `Z_k = effect_k / se_k`, signed so **positive is better**,
and `t_k` is the information fraction. The B-values `B_k = Z_k · sqrt(t_k)` are a
Brownian motion with drift, and everything here is arithmetic on that one object.

In [ ]:
import numpy as np

from axiom.core import D, Interval, Unit
from axiom.design import (
    CANONICAL, STOPPED_ESTIMATE_BIAS, Boundary, BoundaryKind, CrossingProbabilities, Decision,
    LookOutcome, LookSchedule, MonitoringPath, OperatingCharacteristics, Side, SpendingFunction,
    StoppingRule, alpha_spending, crossing_probabilities, harm_boundary, information_fractions,
    monitor, obrien_fleming, operating_characteristics, pocock, sample_size, spending,
    STAGEWISE_ORDERING, StoppedEstimate, stagewise_tail, stopped_estimate,
    AnytimeLook, ConfidenceSequence, anytime_p, confidence_sequence, evalue,
    mixture_boundary, tune,
)

from axiom.display import enable, show, table

import sys; sys.path[:0] = ["..", "../.."]  # nbs/ is on the path either way
from _style import AQUA, BLUE, CRITICAL, GOOD, ORANGE, annotate, band, caption, curve_band, figure, lines, mark_x, mark_y, points

enable();  # every axiom result renders itself from here on

ALPHA, LOOKS = 0.05, 5
information = tuple((k + 1) / LOOKS for k in range(LOOKS))
print("information fractions:", information)
print("from cumulative enrollment:", information_fractions([48, 96, 144, 192, 240]))
print("...against a planned total of 300:", information_fractions([48, 96, 144, 192, 240], 300))

## 1. Three shapes for the same alpha

`pocock` judges every look at the same threshold; `obrien_fleming` uses `c / sqrt(t)`,
so the first look is nearly unstoppable. Both solve their one constant so the *total*
error under the null is exactly `alpha` — the reported constants are the published
ones (2.413 and 2.040 at five equally spaced looks, against the fixed-sample 1.960).
`alpha_spending` instead solves each threshold in turn against a budget, which is what
lets a committee move a look without invalidating the design.

In [ ]:
shapes = {
    "pocock": pocock(ALPHA, information),
    "obrien_fleming": obrien_fleming(ALPHA, information),
    "lan_demets_obf": alpha_spending(ALPHA, information, family="obrien_fleming"),
    "lan_demets_pocock": alpha_spending(ALPHA, information, family="pocock"),
}
table(
    [
        [name, " ".join(f"{v:.3f}" for v in boundary.z),
         f"{boundary.nominal_alpha(LOOKS - 1):.4f}",
         str([round(s, 4) for s in boundary.spent])]
        for name, boundary in shapes.items()
    ],
    headers=("shape", "z at each look", "final nominal p", "cumulative alpha spent"),
)

In [ ]:
fig = lines(
    information,
    {name: boundary.z for name, boundary in shapes.items() if "lan_demets" not in name},
    colors=(BLUE, ORANGE),
    title="Two ways to spend the same 5%",
    subtitle="the Z a look has to beat to stop the study, at five equally spaced looks",
    x_title="information fraction", y_title="critical Z",
)
mark_y(fig, 1.96, text="fixed-sample 1.96")
caption(fig, "O'Brien–Fleming makes the first look nearly unstoppable and finishes close to "
             "the fixed-sample threshold; Pocock judges every look alike and pays for it at "
             "the end. Neither is more correct — they buy different things, and the next "
             "figure is the price list.")

In [ ]:
spent_curves = {name: list(b.spent) for name, b in shapes.items() if "lan_demets" not in name}
fig = lines(
    information, spent_curves,
    colors=(BLUE, ORANGE),
    title="…and when they spend it",
    subtitle="cumulative type-I error committed by the end of each look",
    x_title="information fraction", y_title="alpha spent",
)
mark_y(fig, ALPHA, text="the whole budget")
caption(fig, "Pocock has spent two-thirds of its error budget before the study is half done. "
             "That is what buys its early stopping, and what leaves it least to work with at "
             "the final analysis.")

The spending functions themselves, evaluated anywhere in `[0, 1]`, are what the
thresholds are solved against. `"power"` with `rho` is the tunable family between them.

In [ ]:
grid = (0.1, 0.25, 0.5, 0.75, 1.0)
rows = []
for kind in ("obrien_fleming", "pocock", "power"):
    kind_: SpendingFunction = kind
    rows.append([kind, *[f"{spending(kind_, t, ALPHA):.4f}" for t in grid]])
rows.append(["power (rho=3)", *[f"{spending('power', t, ALPHA, rho=3.0):.4f}" for t in grid]])
table(rows, headers=("spending function", *[f"t={t}" for t in grid]))

## 2. What a shape costs

Every look bought is paid for at the final analysis. Pin the drift the design is
powered for — for a fixed-sample two-arm test at 80 % power and two-sided 5 %, the
expected Z at full information is 2.802 — and read `operating_characteristics` for
the power each shape actually delivers and the information it expects to use.

In [ ]:
ss = sample_size(effect=8.0, sd=14.0, power=0.8)
drift = 8.0 / (14.0 * np.sqrt(4.0 / ss.n))
print(f"fixed-sample n = {ss.n} at 80% power; drift at full information = {drift:.3f}")

schedule = LookSchedule(labels=tuple(f"look_{k + 1}" for k in range(LOOKS)), information=information)
rows = []
for name, boundary in shapes.items():
    rule = StoppingRule(name=name, looks=schedule, boundaries=(boundary,))
    null = operating_characteristics(rule, 0.0)
    alt = operating_characteristics(rule, drift)
    rows.append(
        [name, f"{null.crossings.cumulative('efficacy'):.4f}",
         f"{alt.crossings.cumulative('efficacy'):.3f}",
         f"{alt.expected_information:.3f}", f"{alt.expected_looks:.2f}"]
    )
table(rows, headers=("shape", "type I", "power", "E[information]", "E[looks]"))

In [ ]:
oc_by_shape = {}
for name, boundary in shapes.items():
    alt = operating_characteristics(StoppingRule(name=name, looks=schedule, boundaries=(boundary,)), drift)
    oc_by_shape[name] = (alt.expected_information, alt.crossings.cumulative("efficacy"))

fig = points(
    {name: ([v[0]], [v[1]]) for name, v in oc_by_shape.items()},
    size=13,
    title="The trade, drawn",
    subtitle="power against expected information used, at the drift the study is powered for",
    x_title="expected information used", y_title="power",
    height=420,
)
for name, (info_used, power_) in oc_by_shape.items():
    fig.add_annotation(x=info_used, y=power_, text=name, showarrow=False, yshift=15,
                       font={"size": 11, "color": "#52514e"})
mark_y(fig, 0.8, text="the fixed-sample design's power")
caption(fig, "Up and to the left is better: more power, less enrollment. Nothing dominates — "
             "the shapes sit on a frontier, and which end of it you want is a question about "
             "what an extra period of enrollment costs you.")

Pocock stops soonest — the smallest expected information — and pays for it with the
lowest power, because its final critical value is the largest. O'Brien–Fleming keeps
almost all the fixed-sample power and still stops early when the effect is large.

## 3. A harm boundary is a sentence, not a z

`harm_boundary` takes the rule a monitoring committee writes down — *stop when the
posterior probability that the treatment is worse than the control by more than
`margin` reaches `probability`* — and returns the Z threshold that implements it,
look by look. With `margin = 0` the threshold is constant; a positive margin makes the
early looks *harder* to cross, because a large observed harm is cheap to come by when
the standard error is large.

In [ ]:
se_final = 14.0 * float(np.sqrt(4.0 / ss.n))
flat = harm_boundary(0.95, information)
with_margin = harm_boundary(0.95, information, margin=3.0, se_at_full_information=se_final)
print("se at full information:", round(se_final, 3))
print("margin 0 :", [round(v, 3) for v in flat.z])
print("margin 3 :", [round(v, 3) for v in with_margin.z])
print("detail   :", with_margin.detail)
kind: BoundaryKind = with_margin.kind
side: Side = with_margin.side
print(f"kind={kind} side={side} binding={with_margin.binding}")

A probability rule states a posterior, not an error rate. `crossing_probabilities`
says what it spends: this one stops a *null* study for harm about one time in thirty.

In [ ]:
harm_only = StoppingRule(name="harm_only", looks=schedule, boundaries=(with_margin,))
spent = crossing_probabilities(harm_only, 0.0)
assert isinstance(spent, CrossingProbabilities)
print("P(stop for harm | no true difference) =", round(spent.cumulative("harm"), 4))
print("per look:", [round(p, 4) for p in spent.per_look["harm"]])
print("P(never stop) =", round(spent.continue_probability, 4))

## 4. A rule is not the sum of its boundaries

`StoppingRule` composes an efficacy boundary, a harm boundary and a non-binding
futility boundary into one continuation region per look. Two effects follow, and both
are computed rather than asserted: the efficacy boundary loses some of its own alpha to
paths another boundary stops first, and *ignoring* the non-binding futility boundary —
the conservative convention — overstates the error the rule as run spends. Here the
harm boundary costs the efficacy boundary almost nothing, because the two sit on
opposite sides of zero and a path that reaches one rarely reaches the other; futility
costs it more. Neither is a fact about boundaries in general, which is the point of
computing it.

In [ ]:
efficacy = alpha_spending(ALPHA / 2, information, family="obrien_fleming", side="upper")
futility = Boundary(kind="futility", side="lower", z=(-1.5, -0.8, -0.2, 0.3, 0.8), binding=False)
rule = StoppingRule(name="hyper3", looks=schedule, boundaries=(efficacy, with_margin, futility))
rows = []
for look in range(LOOKS):
    low, high = rule.continuation(look)
    rows.append([look + 1, f"{low:+.3f}", f"{high:+.3f}"])
table(rows, headers=("look", "continue while Z >", "and Z <"))

alone = crossing_probabilities(
    StoppingRule(name="e", looks=schedule, boundaries=(efficacy,)), 0.0
).cumulative("efficacy")
binding = crossing_probabilities(rule, 0.0)
as_run = crossing_probabilities(rule, 0.0, binding_only=False)
print(f"\n{'efficacy alone':22s} {alone:.6f}")
print(f"{'with harm added':22s} {binding.cumulative('efficacy'):.6f}"
      f"   (harm itself takes {binding.cumulative('harm'):.4f})")
print(f"{'as run, futility on':22s} {as_run.cumulative('efficacy'):.6f}"
      f"   (futility takes {as_run.cumulative('futility'):.4f})")
print("binding_only:", binding.binding_only, as_run.binding_only)

## 5. Operating characteristics across the truth

`operating_characteristics` at a grid of drifts is the design's whole story: how often
it stops, for what reason, and how much of the planned enrollment it expects to use.
Note the asymmetry the design was built for — a harmful treatment is stopped far
earlier than a beneficial one is confirmed.

In [ ]:
print(f"{'drift':>7} {'efficacy':>9} {'harm':>7} {'futility':>9} {'E[info]':>8} {'E[looks]':>9}")
rows = []
for d in (-3.5, -2.0, -1.0, 0.0, 1.0, 2.0, 2.802, 3.5):
    oc = operating_characteristics(rule, d, binding_only=False)
    assert isinstance(oc, OperatingCharacteristics)
    c = oc.crossings
    rows.append(
        [f"{d:.2f}", f"{c.cumulative('efficacy'):.3f}", f"{c.cumulative('harm'):.3f}",
         f"{c.cumulative('futility'):.3f}", f"{oc.expected_information:.3f}",
         f"{oc.expected_looks:.2f}"]
    )
table(rows, headers=("drift", "efficacy", "harm", "futility", "E[information]", "E[looks]"))

In [ ]:
drifts = np.linspace(-3.5, 3.5, 29)
by_reason = {"efficacy": [], "harm": [], "futility": []}
expected_info = []
for d in drifts:
    oc = operating_characteristics(rule, float(d), binding_only=False)
    for reason in by_reason:
        by_reason[reason].append(oc.crossings.cumulative(reason))
    expected_info.append(oc.expected_information)

fig = lines(
    drifts, by_reason,
    colors=(GOOD, CRITICAL, AQUA),
    title="What the rule does, against every truth it might face",
    subtitle="probability of stopping for each reason as the true drift varies",
    x_title="true drift", y_title="probability",
)
mark_x(fig, 0.0, text="no true effect")
caption(fig, "The two sides are not mirror images. A strongly harmful treatment is stopped "
             "with near-certainty — mostly by the futility boundary, which sits above the harm "
             "boundary at the early looks — while a strongly beneficial one is confirmed 93% "
             "of the time. Reading this figure is how a monitoring committee finds out what "
             "they actually agreed to.")

In [ ]:
fig = curve_band(
    drifts, expected_info,
    label="expected information used",
    title="…and what it costs to run",
    subtitle="expected information fraction at stopping, across the same drifts",
    x_title="true drift", y_title="expected information used",
)
mark_x(fig, 0.0, text="no true effect")
mark_y(fig, 1.0, text="full enrollment")
caption(fig, "The longest-running studies are not the null ones — futility stops those — but "
             "the ones with a modest positive effect: too small to cross the efficacy boundary "
             "early, too promising to abandon. That peak, not the best case, is the enrollment "
             "a budget should be built on.")
harmful = operating_characteristics(rule, -2.802, binding_only=False)
print("\nstop probability at drift -2.802:", round(harmful.stop_probability, 4))
print("stops by look:", [round(p, 3) for p in harmful.crossings.by_look()])

## 6. Running one study against the rule

`monitor` takes the realized `Z` at each look taken so far. It stops at the first
crossing, and the outermost boundary wins when two fire at once — a statistic below
the harm boundary is a harm stop even though it is also below futility.

In [ ]:
observed_z = [-0.42, -0.55, -3.05, 0.0, 0.0]
effects = [-1.8, -2.0, -9.6, 0.0, 0.0]
ses = [4.3, 3.7, 3.1, 2.7, 2.4]
path = monitor(rule, observed_z[:3], effects=effects[:3], ses=ses[:3])
assert isinstance(path, MonitoringPath)
rows = []
for outcome in path.looks:
    assert isinstance(outcome, LookOutcome)
    low, high = rule.continuation(outcome.look)
    rows.append(
        [outcome.label, f"{outcome.information:.2f}", f"{outcome.z:+.3f}",
         f"({low:+.2f}, {high:+.2f})", outcome.decision]
    )
table(rows, headers=("look", "information", "Z", "continuation", "decision"))

In [ ]:
looks_axis = np.arange(1, LOOKS + 1)
lower = [rule.continuation(k)[0] for k in range(LOOKS)]
upper = [rule.continuation(k)[1] for k in range(LOOKS)]
fig = figure(
    title="One study, walked against the rule",
    subtitle="the continuation region at each look, and the Z that was actually observed",
    x_title="look", y_title="Z (positive is better)",
)
band(fig, looks_axis, lower, upper, name="keep going", color=BLUE)
fig.add_scatter(x=looks_axis[:3], y=observed_z[:3], mode="lines+markers",
                line={"color": CRITICAL, "width": 2}, marker={"size": 10, "color": CRITICAL},
                name="observed Z", showlegend=True)
annotate(fig, 3, observed_z[2], f"stopped: {path.decision}")
caption(fig, "The shaded region is where the study keeps going. The third look leaves it "
             "downwards — through the harm boundary rather than the futility boundary, "
             "because when two fire at once the outermost one wins.")
decision: Decision = path.decision
print(f"\ndecision={decision} at look {path.stopped_at + 1}, threshold {path.threshold():.3f}")
print("crossed the boundary:", rule.of_kind("harm") is rule.crossings(2, -3.05)[0])

The number that leaves the study carries the reason it is suspect. `monitor` emits a
`LedgerLine` whose assumption is `stopped_estimate_bias` when a boundary stopped it and
`canonical_joint_distribution` when it did not — rule 4 of the repo, applied to the one
estimate a sequential design is most likely to over-read.

In [ ]:
line = path.ledger_line()
print(line.kind, "|", line.statement)
print("assumption:", line.assumption.name, "-", line.assumption.statement)
print("challenged by:", line.assumption.challenged_by)
print("detail:", line.detail)
print("\nstill running:", monitor(rule, [0.1, 0.4]).ledger_line().assumption.name)
print("named assumptions:", CANONICAL.name, "|", STOPPED_ESTIMATE_BIAS.name)

## 7. Looks that did not happen when they were planned

The point of a spending function: recompute the remaining thresholds against the
information that actually accrued. Enrollment ran slow, so the third look happens at
52 % rather than 60 % — only the thresholds from there on move.

In [ ]:
planned = information
actual = (0.2, 0.4, 0.52, 0.79, 1.0)
rows = []
for label, fractions in (("planned", planned), ("actual", actual)):
    b = alpha_spending(ALPHA / 2, fractions, family="obrien_fleming", side="upper")
    rows.append(
        [label, str([round(t, 2) for t in fractions]), str([round(v, 3) for v in b.z]),
         f"{b.spent[-1]:.4f}"]
    )
table(rows, headers=("schedule", "information fractions", "z", "total spent"))

## 8. The dimension the statistic is not carrying

`Z` is dimensionless by construction — an effect divided by its standard error — which
is why one module serves an outcome measured in mmHg and one measured in anything else.
The units live with the effect, not with the boundary.

In [ ]:
mmhg = Unit(name="mmHg", dimension=D.outcome)
print("outcome unit:", mmhg.name, "| dimension:", mmhg.dimension)
print("effect at the stopping look:", path.looks[-1].effect, mmhg.name)
print("its se:", path.looks[-1].se, mmhg.name, "-> Z =", round(path.looks[-1].z, 3), "(dimensionless)")

## 9. The number the study is allowed to report

Section 6 ended with a ledger line saying the stopped estimate is biased. Saying so is
not fixing it, and a number that travels into `calibrate` and `meta` under a warning is
a number that gets pooled anyway. `stopped_estimate` is the correction the assumption's
own `challenged_by` asks for: the drift whose stage-wise tail probability is one half,
with the interval that inverts the same tail.

The path above stopped for harm at the third look, so the naive estimate is biased
*downward* — too good a case against the treatment — and the correction moves toward
the null. It moves a long way, and the reason is worth reading off the boundaries: the
lower edge of the continuation region at the first two looks is the **futility**
boundary at −1.5 and −0.8, not the harm boundary at −2.12 and −2.31. A study with a
drift of −3.94 would almost certainly have been abandoned for futility before it ever
reached the third look. Surviving to look 3 is therefore strong evidence the drift is
not −3.94, and the corrected interval spans zero where the naive one does not.

In [ ]:
corrected = stopped_estimate(path)
assert isinstance(corrected, StoppedEstimate)

table(
    [["drift", f"{corrected.naive_drift:.4f}", f"{corrected.drift:.4f}",
      corrected.drift_interval.text()],
     ["effect (mmHg)", f"{corrected.naive_effect:.4f}", f"{corrected.effect:.4f}",
      corrected.effect_interval.text()]],
    headers=("scale", "naive", "median-unbiased", "stage-wise interval"),
    title=f"{corrected.rule}, stopped for {corrected.crossed} at look {corrected.look + 1}",
)
print("correction:", round(-corrected.bias, 4), "on the drift scale;",
      round(-corrected.effect_bias, 4), "mmHg")
print("stopped early:", corrected.stopped_early)

### Where the correction comes from, and where it is nothing

The tail the estimate inverts is a probability, so it can be printed. `stagewise_tail`
is that probability at any drift; it is strictly increasing, and the three points where
it hits 0.025, 0.5 and 0.975 are the interval and the estimate.

The size of the correction is not a constant of the method — it is what conditioning on
*not having stopped earlier* is worth, and every boundary the rule carries does that
conditioning, binding or not. A non-binding futility boundary is invisible to the error
arithmetic and not to the estimate: the study really did keep going past it.

At the first look nothing precedes, so there is nothing to condition on and the
corrected estimate equals the naive one exactly. Under a rule whose early boundaries are
almost never crossed, there is almost nothing to condition on either.

In [ ]:
grid = [-2.0, 0.0, 2.0, 4.0, 6.0]
print("stage-wise tail at look 3, z = -3.05, by drift:")
print("  ", [round(stagewise_tail(path.rule, 2, -3.05, d), 4) for d in grid])

four = LookSchedule(labels=("L1", "L2", "L3", "L4"), information=(0.25, 0.5, 0.75, 1.0))
rows = []
for name, boundary in (("Pocock", pocock(ALPHA, four)), ("O'Brien-Fleming", obrien_fleming(ALPHA, four))):
    shape = StoppingRule(name=name, looks=four, boundaries=(boundary,))
    corrections = []
    for k in range(4):
        result = stopped_estimate(monitor(shape, [0.5] * k + [2.5]))
        corrections.append(f"{result.bias:+.3f}" if isinstance(result, StoppedEstimate) else "no stop")
    rows.append([name, str([round(v, 2) for v in boundary.z])] + corrections)
table(rows, headers=("rule", "thresholds", "look 1", "look 2", "look 3", "look 4"),
      title="what a stop at z = 2.5 is corrected by")
print("\nassumption on the corrected line:", STAGEWISE_ORDERING.name,
      "| challenged by:", STAGEWISE_ORDERING.challenged_by)
print(corrected.ledger_line().statement)

## 10. The look nobody scheduled

Everything above prices looks that were **fixed in advance**. That is the right machinery for
a trial with a protocol and a monitoring committee. It is not what happens to an experiment
inside a company: the dashboard is open, the interval refreshes hourly, and somebody decides
the first afternoon it looks good. A spending function cannot price a look nobody scheduled,
and a fixed-sample interval read every day is not a 95 % interval.

Read at ten looks, a nominal 95 % Wald interval excludes the truth about a fifth of the time.
That is not a subtle inflation — it is four times the rate it advertises.

In [ ]:
from scipy import stats as _st

t_grid = np.linspace(0.1, 1.0, 10)
dt = np.diff(np.concatenate([[0.0], t_grid]))
rng_a = np.random.default_rng(0)
paths = np.cumsum(rng_a.normal(0.0, np.sqrt(dt), size=(20_000, t_grid.size)), axis=1)

rho = tune(ALPHA, 1.0)
boundary = np.asarray([mixture_boundary(x, alpha=ALPHA, rho=rho) for x in t_grid])
naive = float(_st.norm.isf(ALPHA / 2)) * np.sqrt(t_grid)

table([["fixed-sample Wald, read once", f"{(np.abs(paths[:, -1]) > naive[-1]).mean():.4f}"],
       ["fixed-sample Wald, read at all ten looks",
        f"{(np.abs(paths) > naive).any(axis=1).mean():.4f}"],
       ["confidence sequence, read at all ten looks",
        f"{(np.abs(paths) > boundary).any(axis=1).mean():.4f}"]],
      headers=("what was read", "P(ever excluded the truth)"),
      title=f"20,000 null paths, nominal alpha {ALPHA}")

A **confidence sequence** is an interval at every possible time such that the probability the
*whole sequence* ever excludes the truth is at most `alpha`. It is built from a martingale:
mixing `exp(λ(B_t − θt) − λ²t/2)` over `λ ~ N(0, 1/ρ)` has a closed form, starts at 1, and
Ville's inequality does the rest. Inverting it gives a boundary that grows like `sqrt(t log t)`
where the fixed-sample one grows like `sqrt(t)` — and that gap *is* the peeking.

There is no free anytime validity. The interval is wider at every single look, by about half
again at `alpha = 0.05`, and `ρ` decides where it is tightest. `tune` picks it numerically for
a target information fraction rather than by a rule of thumb, and the choice is recorded.

In [ ]:
rows = []
for target in (0.25, 1.0):
    tuned = tune(ALPHA, target)
    rows.append([f"tuned at t = {target}", f"{tuned:.4f}"] +
                [f"{mixture_boundary(t, alpha=ALPHA, rho=tuned) / t:.3f}"
                 for t in (0.25, 0.5, 1.0)])
rows.append(["fixed-sample Wald", "—"] +
            [f"{float(_st.norm.isf(ALPHA / 2)) / np.sqrt(t):.3f}" for t in (0.25, 0.5, 1.0)])
table(rows, headers=("sequence", "rho", "half-width at 0.25", "at 0.5", "at 1.0"),
      title="the price of being allowed to look whenever you like (drift scale)")

### One study, read whenever

`confidence_sequence` takes the same realized statistics `monitor` does and needs no schedule,
because it does not have one. Every look gets an interval labelled `anytime` — not `wald`,
because they are not the same object and gate 6 exists so they cannot be confused — and an
**e-value**: the martingale itself, whose reciprocal is a p-value valid however often it is
read. The running maximum is the one to quote, because that is what Ville's inequality bounds.

In [ ]:
walk = [0.4, 1.2, 2.1, 2.9]
fractions = [0.25, 0.5, 0.75, 1.0]
seq: ConfidenceSequence = confidence_sequence(
    walk, fractions, alpha=ALPHA, name="hyper3-anytime", labels=["w6", "w12", "w18", "w24"])
table([[l.label, f"{l.information:.2f}", f"{l.z:+.2f}", f"{l.drift:+.3f}",
        l.interval.text(), f"{l.evalue:.3g}", f"{l.running_evalue:.3g}", str(l.crossed)]
       for l in seq.looks],
      headers=("look", "t", "Z", "drift", "anytime interval", "e-value", "running", "crossed"))
print(seq.summary())

The last look is the point. `Z = 2.9` at full information is comfortably "significant" by a
fixed-sample reading — a Wald interval on the drift excludes zero — and the anytime interval
does not. Both are correct. They answer different questions, and only one of them is the
question somebody who has been watching a dashboard for six months is entitled to ask.

`AnytimeLook` and the sequence are `Spec`s, so a readout carries its own ledger line, and the
assumption left standing is the one every sequential result here rests on: the canonical joint
distribution.

In [ ]:
final: AnytimeLook = seq.final
wald_interval = Interval(lower=final.drift - 1.96, upper=final.drift + 1.96,
                         definition="wald", mass=0.95)
table([["fixed-sample Wald", wald_interval.text(), str(not wald_interval.contains(0.0))],
       ["confidence sequence", final.interval.text(), str(final.crossed)]],
      headers=("interval", "at the final look", "excludes zero"))
print("anytime p =", round(final.anytime_p, 4), "| fixed-sample p =",
      round(float(2 * _st.norm.sf(abs(final.z))), 4))
print()
show(seq.ledger_line())
print("width against a fixed-sample interval where it is tuned:",
      round(seq.width_ratio(seq.tuned_at), 3), "x")
print("anytime_p as a bare function:", round(anytime_p(2.9, 1.0, rho=seq.rho), 4))

## What this notebook decided

- The shape is a trade, and it is priced: Pocock stops soonest and has the least power,
  O'Brien–Fleming keeps nearly the fixed-sample power and still stops early when the
  effect is large. `operating_characteristics` gives both numbers for both.
- A posterior-probability harm rule is a legitimate boundary, and
  `crossing_probabilities` says what it spends. Stating a posterior is not the same as
  controlling an error rate, and the design should know both numbers.
- Boundaries interact. Adding a harm boundary takes paths away from the efficacy
  boundary; treating futility as non-binding overstates what the rule as run spends.
  `binding_only` makes the gap a number.
- An estimate reported at the look that stopped the study is biased away from the null,
  and the ledger line says so rather than letting the number travel alone.
- Saying so is not fixing it. `stopped_estimate` inverts the stage-wise ordered tail for
  the median-unbiased estimate and its interval, and replaces the warning with a
  correction — larger the later the stop, and exactly zero at the first look, where
  there is nothing to condition on.
- All of that prices looks that were planned. For the ones that were not, a confidence
  sequence is valid at every time at once — at the cost of being about half again as wide
  at any one of them, which is the honest price of a dashboard nobody closes.